This notebook process one-by-one the netcdf files in `data/SWOT_L3_LR_SSH_Expert_v3_0_cg_inversion` and `data/SWOT_L3_LR_SSH_Expert_v3_0`.

For each file it:
- checks if the corresponding file is already present in the directory `data/SWOT_L3_LR_SSH_Expert_v3_0_cg_imbalance`,
- if it is not present, it:
    - computes the $u$ and $v$ imbalance components for ,
    - creates a new dataset with the same dimensions and coordinates as the original one but containing only the variables ``u_imb`` and ``v_imb``,
    - stores it in the directory `data/SWOT_L3_Expert_v3_0_imbalance` using the original filename.

In [ ]:
import glob
import os
import time
import warnings

import jax
jax.config.update("jax_enable_x64", True)
import numpy as np
import xarray as xr

from jaxparrow.cyclogeostrophy import cyclogeostrophic_imbalance

In [ ]:
warnings.filterwarnings("ignore")

INPUT_CG_DIR = "data/SWOT_L3_LR_SSH_Expert_v3_0_cg_inversion"
INPUT_G_DIR = "data/SWOT_L3_LR_SSH_Expert_v3_0"
OUTPUT_DIR = "data/SWOT_L3_LR_SSH_Expert_v3_0_cg_imbalance"

while True:
    for filepath_cg in sorted(glob.glob(os.path.join(INPUT_CG_DIR, "*.nc"))):
        filename = os.path.basename(filepath_cg)
        output_path = os.path.join(OUTPUT_DIR, filename)

        if os.path.exists(output_path):
            try:
                ds_out = xr.open_dataset(output_path)
                ds_out.close()
                continue
            except Exception:
                pass

        try:
            filepath_g = os.path.join(INPUT_G_DIR, filename)

            ds_g, ds_cg = map(xr.open_dataset, [filepath_g, filepath_cg])

            ds_g, ds_cg = map(lambda ds: ds.where(np.abs(ds.latitude) > 10), [ds_g, ds_cg])

            lat = ds_g.latitude.values
            lon = ds_g.longitude.values

            ug = ds_g.ugos_filtered.values
            vg = ds_g.vgos_filtered.values

            ucg_fp = ds_cg.ucg_fp.values
            vcg_fp = ds_cg.vcg_fp.values
            ucg_gw = ds_cg.ucg_gw.values
            vcg_gw = ds_cg.vcg_gw.values
            u_cg_mb = ds_cg.ucg_mb.values
            v_cg_mb = ds_cg.vcg_mb.values

            (imb_ug, imb_vg), (imb_ucg_fp, imb_vcg_fp), (imb_ucg_gw, imb_vcg_gw), (imb_ucg_mb, imb_vcg_mb) = map(
                lambda args: cyclogeostrophic_imbalance(ug, vg, args[0], args[1], lat, lon),
                [(ug, vg), (ucg_fp, vcg_fp), (ucg_gw, vcg_gw), (u_cg_mb, v_cg_mb)]
            )

            ds_out = xr.Dataset(
                {
                    "imb_ug": xr.DataArray(imb_ug, dims=ds_g.ugos_filtered.dims, coords=ds_g.ugos_filtered.coords),
                    "imb_vg": xr.DataArray(imb_vg, dims=ds_g.vgos_filtered.dims, coords=ds_g.vgos_filtered.coords),
                    "imb_ucg_fp": xr.DataArray(imb_ucg_fp, dims=ds_cg.ucg_fp.dims, coords=ds_cg.ucg_fp.coords),
                    "imb_vcg_fp": xr.DataArray(imb_vcg_fp, dims=ds_cg.vcg_fp.dims, coords=ds_cg.vcg_fp.coords),
                    "imb_ucg_gw": xr.DataArray(imb_ucg_gw, dims=ds_cg.ucg_gw.dims, coords=ds_cg.ucg_gw.coords),
                    "imb_vcg_gw": xr.DataArray(imb_vcg_gw, dims=ds_cg.vcg_gw.dims, coords=ds_cg.vcg_gw.coords),
                    "imb_ucg_mb": xr.DataArray(imb_ucg_mb, dims=ds_cg.ucg_mb.dims, coords=ds_cg.ucg_mb.coords),
                    "imb_vcg_mb": xr.DataArray(imb_vcg_mb, dims=ds_cg.vcg_mb.dims, coords=ds_cg.vcg_mb.coords),
                },
            )
            ds_out.to_netcdf(output_path)

            ds_g.close()
            ds_cg.close()
            ds_out.close()
        except Exception:
            print(f"Error for file {filename}")
    
    time.sleep(1)